In [ ]:
import math
import random
class value:
    def __init__(self,data,_children=(),_op='leaf node bruh'):
        self.data=data
        self._prev=set(_children)
        self.grad=0
        self._op=_op
        self._backward = lambda: None
    def __repr__(self):
        return f"value(data={self.data})"
    def __add__(self,other):
        other = other if isinstance(other, value) else value(other)
        out=value(self.data+other.data,(self,other),'+')
      
        
        def _backward():
            self.grad+= 1 * out.grad
            other.grad+= 1* out.grad
        out._backward=_backward
        
        return out

    def __radd__(self, other):   # lets  2 + a  work, not just a + 2
        return self + other

    def __neg__(self):
        return self * value(-1)

    def __sub__(self, other):
        return self + (-other)

    
    def __mul__(self,other):
        other = other if isinstance(other, value) else value(other)
        out=value(self.data*other.data,(self,other),'*')

        def _backward():
            self.grad+=other.data*out.grad
            other.grad+=self.data*out.grad
        out._backward=_backward    

        
        return out

    def __rmul__(self, other):   # lets  2 * a  work, not just a * 2
        return self * other

    def __pow__(self, other):
        out = value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    
    def tanh(self):
        x=self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = value(t, (self,), 'tanh')

        def _backward():
            self.grad+=(1-t**2)*out.grad
        out._backward = _backward     
        return out


    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [465]:
a=value(2)
b=value(7)
c=value(10)
print(a)
print(c)
print(b)

value(data=2)
value(data=10)
value(data=7)


In [466]:

x = (a)
print(x.grad)

0


In [467]:

print(b.tanh())
d = a + b        # d IS the out that __add__ made
print(d.data)    # reads out.data
print(d.grad) 
d.grad=1

value(data=0.9999983369439447)
9
0


In [468]:
print(d._backward())

None


In [469]:
print(a.grad)
print(b.grad)

1
1


In [470]:
a = value(2.0)
b = value(-3.0)
e = a * b
e.grad = 1.0        
e._backward()
print("a.grad:", a.grad)
print("b.grad:", b.grad)

a.grad: -3.0
b.grad: 2.0


In [471]:
class Nueron:
    def __init__(self,nin):
        self.w=[value(random.uniform(-1,1)) for _ in range(nin)]
        self.b=value(random.uniform(-1,1))

    def __call__(self,x):
        act=sum((wi*xi for wi,xi in zip(self.w,x)),self.b)
        out=act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [472]:
class Layer:
    def __init__(self,nin,nout):
        self.neurons=[Nueron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs=[n(x) for n in self.neurons]
        return outs
    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params

In [473]:
class MLP:
    def __init__(self,nin,nouts):
        sz=[nin]+nouts
        self.layers=[Layer(sz[i],sz[i+1]) for i in range(len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x=layer(x)
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

In [474]:
m = MLP(3, [4,4,1])

In [475]:
xs = [
    [value(2.0), value(3.0), value(-1.0)],
    [value(3.0), value(-1.0), value(0.5)],
    [value(0.5), value(1.0), value(1.0)],
    [value(1.0), value(1.0), value(-1.0)],
]
ys = [1.0, -1.0, -1.0, 1.0]



In [476]:
ypred = [m(x)[0] for x in xs]
print(ypred)

[value(data=0.29087790454857543), value(data=0.4222127101766813), value(data=0.610280597475418), value(data=0.31745697969401643)]


In [477]:
for p in m.parameters():
    p.data -= 0.01 * p.grad

In [478]:
loss = sum(((yout - value(ygt))**2 for ygt, yout in zip(ys, ypred)), value(0))
print(loss)

value(data=5.584411716419724)


In [479]:
loss.backward()
print(m.layers[0].neurons[0].w[0])

value(data=0.41671507096529825)


In [480]:
print(len(m.parameters()))

41


In [481]:
xs = [
    [value(2.0), value(3.0), value(-1.0)],
    [value(3.0), value(-1.0), value(0.5)],
    [value(0.5), value(1.0), value(1.0)],
    [value(1.0), value(1.0), value(-1.0)],
]
ys = [1.0, -1.0, -1.0, 1.0]

m = MLP(3, [4,4,1])

ypred = [m(x)[0] for x in xs]
loss = sum(((yout - value(ygt))**2 for ygt, yout in zip(ys, ypred)), value(0))
print("loss before:", loss)

for p in m.parameters():
    p.grad = 0.0

loss.backward()

for p in m.parameters():
    p.data -= 0.01 * p.grad

ypred = [m(x)[0] for x in xs]
loss = sum(((yout - value(ygt))**2 for ygt, yout in zip(ys, ypred)), value(0))
print("loss after:", loss)

loss before: value(data=7.027650496305459)
loss after: value(data=6.804812339589574)


In [527]:
for k in range(20):
    # forward pass
    ypred = [m(x)[0] for x in xs]
    loss = sum(((yout - value(ygt))**2 for ygt, yout in zip(ys, ypred)), value(0))

    # zero grad
    for p in m.parameters():
        p.grad = 0.0

    # backward pass
    loss.backward()

    # update
    for p in m.parameters():
        p.data -= 0.1 * p.grad

    print(k, loss.data)

0 0.011521518349297223
1 0.011091023131908354
2 0.010690882134330212
3 0.010318021208223551
4 0.009969765870568987
5 0.009643778582643935
6 0.009338007458630166
7 0.009050644048832036
8 0.008780088380628798
9 0.008524919844036931
10 0.008283872814375443
11 0.008055816137805514
12 0.007839735784973375
13 0.0076347201170737684
14 0.007439947317200422
15 0.007254674625135746
16 0.00707822908115451
17 0.006909999538041345
18 0.0067494297434110475
19 0.006596012328911738


In [529]:
print(ypred)
print(ys)

[value(data=0.9794998174095058), value(data=-0.9646994697451867), value(data=-0.9494180854797286), value(data=0.951306085289216)]
[1.0, -1.0, -1.0, 1.0]
